# M2 — DeepAR on Colab (the first rung in the heavy environment)

M0 (seasonal-naive) and M1 (ARIMA) run on a laptop. **M2 DeepAR** is the first model that needs the *heavy group* (GluonTS + PyTorch Lightning), so we run it on Colab.

This notebook is a thin driver — all the logic lives in the repo (`src/models/deepar.py`, `experiments/run_deepar.py`). It clones the repo, installs the heavy group, lets the data auto-download, trains DeepAR on the **Exchange** train split, scores the **test** windows with the *same* metrics module as M0/M1, and appends one row to `results/registry.csv`.

**Before you start:** `Runtime → Change runtime type → GPU` (Exchange is tiny, so CPU also works in a couple of minutes; the runner uses `--accelerator auto`, which picks the GPU when present).

## 1. Clone the sandbox repo

In [ ]:
!git clone -q https://github.com/Icaica14/pml-diffusion-tsf-test.git
%cd pml-diffusion-tsf-test

## 2. Install the heavy group

`gluonts[torch]` pulls a compatible PyTorch + Lightning. **Keep Colab's preinstalled numpy** — do *not* downgrade it. Colab's pandas/scipy are compiled against numpy 2.x, so forcing `numpy<2.0` breaks the binary ABI (you get `numpy.dtype size changed, Expected 96 ... got 88`). Our code is numpy-2 safe, so we use whatever numpy Colab ships. This takes a minute or two.

In [ ]:
!pip install -q "gluonts[torch]"

In [ ]:
# Sanity check: versions + whether a GPU is visible.
import gluonts, torch, numpy as np
print('gluonts', gluonts.__version__, '| torch', torch.__version__, '| numpy', np.__version__)
print('CUDA available:', torch.cuda.is_available())

## 3. Train + evaluate M2

The Exchange series auto-downloads on first use (the runner's `download()` is idempotent). Training is one global DeepAR over the 8 channels; the test windows are then forecast with no refit (each window conditioned only on its own context — the same leakage-free protocol as M1).

`--epochs 30` is a reasonable budget for Exchange; bump it if the loss is still falling.

In [ ]:
!python -m experiments.run_deepar --epochs 30 --samples 100 --accelerator auto

## 3b. Tuned variant — DeepAR without calendar features

The baseline DeepAR above underperforms M0/M1 on Exchange. One concrete suspect: the series `start` is a **placeholder** date (1990-01-01) — Exchange is not truly calendar-aligned — so GluonTS's default *daily time features* are spurious covariates the LSTM can over-fit.

This cell re-runs DeepAR with `--no-time-features` (`time_features=[]`), an otherwise-identical clean ablation (same epochs / seed). It appends a separate **`deepar_notf`** row so you can compare it head-to-head with the baseline `deepar` row in the next section.

In [ ]:
!python -m experiments.run_deepar --epochs 30 --samples 100 --no-time-features --accelerator auto

## 4. Inspect the results and bring the rows home

The cloned repo's `results/registry.csv` already holds the committed M0, M1 and baseline **`deepar`** rows; the cells above appended a fresh `deepar` (identical — it's deterministic) and the tuned **`deepar_notf`**. Read it back to compare all rungs side by side, then download the file and send it over to record (duplicate `deepar` rows are harmless — they get de-duplicated on commit).

In [ ]:
import pandas as pd
df = pd.read_csv('results/registry.csv')
cols = ['model', 'MASE', 'CRPS', 'cov50', 'cov90', 'width90', 'fit_s', 'predict_s']
print(df[[c for c in cols if c in df.columns]].to_string(index=False))

In [ ]:
# Download the updated registry so you can commit it from your laptop.
from google.colab import files
files.download('results/registry.csv')